## Flow Shop Scheduling Problem (FSP) 简介

**工艺路线**：与 Job Shop 不同，Flow Shop 中所有工件共享完全相同的机器加工顺序。
每个工件必须依次经过 M1 → M2 → ... → Mm 进行加工。

**特点**：FSP 本质上是一个排列排序问题——确定了 Job 的加工顺序后，各机器上的作业次序也随之确定，无需额外排程决策。

**约束**：同一时刻一台机器只能加工一个工件的一道工序；
同一工件的工序必须按 M1→M2→...→Mm 的顺序依次加工。

**目标**：最小化最大完工时间 (Makespan)。

FSP 是三类车间调度问题中结构最简单的一种，但其求解同样属于 NP-hard 范畴。

## GA 求解 FSP 算法概述

由于 FSP 中所有工件共享相同的机器顺序，GA 的编码与 JSP 相同（基于工序的编码），区别仅在于 J 矩阵的每行都相同（所有工件经过相同的机器序列）。

算法流程：
1. **编码**：基于工序的染色体序列（与 JSP 完全一致）
2. **初始化**：随机生成种群
3. **解码**：转化为调度方案并计算 Makespan
4. **选择**：锦标赛选择
5. **交叉**：POX 交叉
6. **变异**：逆序变异
7. **迭代**：重复 3-6 直到满足终止条件

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import copy

### 编码策略

与 JSP 完全一致的基于工序的编码：染色体长度为 n×m，每个 Job 编号出现 m 次。

**FSP 的 J 矩阵特点**：
由于所有工件经过相同的机器顺序，J 矩阵的每一行完全相同。
例如 5 个工件、5 台机器的 FSP 问题：
```
J = [
    [1, 2, 3, 4, 5],  # Job 0: M1→M2→M3→M4→M5
    [1, 2, 3, 4, 5],  # Job 1: M1→M2→M3→M4→M5
    [1, 2, 3, 4, 5],  # Job 2: M1→M2→M3→M4→M5
    [1, 2, 3, 4, 5],  # Job 3: M1→M2→M3→M4→M5
    [1, 2, 3, 4, 5],  # Job 4: M1→M2→M3→M4→M5
]
```
而 P 矩阵则不同——不同工件的同一道工序加工时间可以不同。

In [ ]:
def createInd(J):
    """
    create individual using operation-based encoding
    (same as JSP: each job appears m times in the chromosome)
    """
    n = J.shape[0]
    s = []
    Jm = J.copy()
    while not np.all(Jm == 0):
        I = np.random.randint(0, n)
        M = Jm[I, 0]
        if M != 0:
            s.append(I)
            b = np.roll(Jm[I, :], -1)
            b[-1] = 0
            Jm[I,:] = b
    return s

def createPop(J, popSize):
    """create population"""
    pop = []
    for i in range(popSize):
        pop.append(createInd(J))
    return pop

### 交叉与变异操作

由于染色体结构与 JSP 完全一致，FSP 使用与 JSP 完全相同的遗传操作：

**POX 交叉**：将 Job 随机分为 S1 和 S2，子代保留一个父代 S1 的位置，另一父代的非 S1 Job 按顺序填充。

**逆序变异**：以概率 pm 反转两个随机位置间的子序列。

与 JSP 不同之处：FSP 的机器顺序固定（J 矩阵每行相同），因此解码时只需关注 Job 的顺序排列，机器侧无需额外决策。

In [ ]:
def choice(fitness, k=3, pool=None):
    """tournament selection"""
    if pool is None:
        pool = len(fitness)
    n = len(fitness)
    result = []
    for _ in range(pool):
        indices = random.sample(range(n), k)
        best = indices[0]
        for idx in indices[1:]:
            if fitness[idx] < fitness[best]:
                best = idx
        result.append(best)
    return result

def crossover(parent1, parent2):
    """
    POX crossover (same as JSP)
    """
    n = max(parent1) + 1
    chrom_len = len(parent1)
    job_set = list(range(n))
    num_extract = random.randint(1, n - 1)
    S1 = set(random.sample(job_set, num_extract))

    child1 = [None] * chrom_len
    temp = [item for item in parent2 if item not in S1]
    idx = 0
    for i in range(chrom_len):
        if parent1[i] in S1:
            child1[i] = parent1[i]
        else:
            child1[i] = temp[idx]
            idx += 1

    child2 = [None] * chrom_len
    temp = [item for item in parent1 if item not in S1]
    idx = 0
    for i in range(chrom_len):
        if parent2[i] in S1:
            child2[i] = parent2[i]
        else:
            child2[i] = temp[idx]
            idx += 1

    return child1, child2

def mutation(chromosome, pm=0.1):
    """
    mutation: reverse a subsequence with probability pm
    """
    if random.random() < pm:
        mutant = chromosome.copy()
        idx1, idx2 = random.sample(range(len(mutant)), 2)
        rl, rr = min(idx1, idx2), max(idx1, idx2)
        mutant[rl:rr] = mutant[rl:rr][::-1]
        return mutant
    return chromosome.copy()

def decode(chromosome, J, P):
    """
    Decode: convert operation-based chromosome to schedule
    In FSP, all jobs share the same machine sequence (each row of J is identical).
    """
    n = J.shape[0]
    job_step = [0] * n
    job_end_time = [0] * n
    machine_num = J.max()
    machine_end_time = [0] * machine_num
    schedule = []

    for job in chromosome:
        op_idx = job_step[job]
        machine = J[job, op_idx] - 1
        p_time = P[job, op_idx]
        start_time = max(job_end_time[job], machine_end_time[machine])
        end_time = start_time + p_time
        schedule.append((job, op_idx, machine, start_time, end_time))
        job_step[job] += 1
        job_end_time[job] = end_time
        machine_end_time[machine] = end_time

    makespan = max(job_end_time)
    return schedule, makespan

### 解码

遍历染色体（Job 序列），对每个 Job 编号确定是第几步工序，
从 J 矩阵获取机器编号（每行相同），从 P 矩阵获取加工时间，
最早开始时间 = max(Job 上一工序完工时间, 机器最后完工时间)。

FSP 的解码与 JSP 在实现上完全一致，因为 J 矩阵每行相同只是在数据层面上的特征，不影响解码逻辑。

In [ ]:
def draw_gantt(schedule, J, P, title="FSP Gantt Chart"):
    """Draw Gantt chart for FSP schedule"""
    n = J.shape[0]
    machine_num = J.max()
    colors = plt.cm.tab10(np.linspace(0, 1, n))
    fig, ax = plt.subplots(figsize=(12, 6))
    machine_ops = {i: [] for i in range(machine_num)}
    for job, op_idx, machine, start, end in schedule:
        machine_ops[machine].append((job, op_idx, start, end))
    for machine in range(machine_num):
        for job, op_idx, start, end in machine_ops[machine]:
            ax.barh(machine, end - start, left=start, height=0.6,
                    color=colors[job], edgecolor="black", linewidth=1)
            ax.text((start + end) / 2, machine, f"J{job}O{op_idx}",
                    ha="center", va="center", fontsize=9, fontweight="bold")
    ax.set_yticks(range(machine_num))
    ax.set_yticklabels([f"M{i+1}" for i in range(machine_num)])
    ax.set_xlabel("Time", fontsize=12)
    ax.set_ylabel("Machine", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.grid(axis="x", alpha=0.3)
    patches = [mpatches.Patch(color=colors[i], label=f"Job {i}") for i in range(n)]
    ax.legend(handles=patches, loc="upper right")
    plt.tight_layout()
    plt.show()

def print_schedule(schedule):
    """Print schedule table"""
    print("Job | Op | Machine | Start | End")
    print("---" * 8)
    for job, op_idx, machine, start, end in schedule:
        print(f" J{job} | {op_idx} |   M{machine+1}  |  {start:.0f}  | {end:.0f}")
    makespan = max(end for _, _, _, _, end in schedule)
    print(f"\nMakespan = {int(makespan)}")

### 主函数流程

`GA_FSP(J, P, popSize, generations, pc, pm)`：
1. 调用 `createPop()` 初始化种群
2. 每代循环：解码所有个体 → 锦标赛选择 → POX 交叉 → 逆序变异
3. 输出收敛曲线和最优调度的甘特图

建议参数：`popSize=100`、`generations=200`、`pc=0.8`、`pm=0.1`。

In [ ]:
def GA_FSP(J, P, popSize=50, generations=100, pc=0.8, pm=0.1):
    """
    GA for Flow Shop Scheduling Problem
    J: machine sequence matrix (n x m), all rows are identical
    P: processing time matrix (n x m)
    """
    n = J.shape[0]
    m = J.shape[1]
    print(f"FSP实例: {n}个Job, {m}道工序, {J.max()}台机器")

    pop = createPop(J, popSize)
    best_makespan = float("inf")
    best_chromosome = None
    best_schedule = None
    best_history = []

    for gen in range(generations):
        fitness = []
        schedules = []
        for chrom in pop:
            schedule, makespan = decode(chrom, J, P)
            fitness.append(makespan)
            schedules.append(schedule)

        min_idx = int(np.argmin(fitness))
        gen_best = fitness[min_idx]
        best_history.append(gen_best)

        if gen_best < best_makespan:
            best_makespan = gen_best
            best_chromosome = pop[min_idx].copy()
            best_schedule = schedules[min_idx]

        selected_idx = choice(fitness, k=3, pool=popSize)
        selected_chroms = [pop[i] for i in selected_idx]

        new_pop = []
        i = 0
        while i < popSize:
            if i + 1 < popSize and random.random() < pc:
                c1, c2 = crossover(selected_chroms[i], selected_chroms[i+1])
                new_pop.append(c1)
                new_pop.append(c2)
                i += 2
            else:
                new_pop.append(selected_chroms[i].copy())
                i += 1

        for i in range(len(new_pop)):
            new_pop[i] = mutation(new_pop[i], pm)

        pop = new_pop

        if (gen + 1) % 50 == 0 or gen == 0:
            print(f"Generation {gen+1:4d}: Best = {gen_best:3d}, Global Best = {best_makespan:3d}")

    print(f"\nFinal Results:")
    print(f"Best Makespan = {best_makespan}")
    print(f"Best Chromosome = {best_chromosome}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    ax1.plot(best_history, "b-", linewidth=1.5)
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Makespan")
    ax1.set_title("Convergence Curve")
    ax1.grid(alpha=0.3)

    draw_gantt(best_schedule, J, P, title=f"Best Schedule (Makespan={best_makespan})")
    print_schedule(best_schedule)

    return best_chromosome, best_makespan, best_schedule

In [ ]:
# FSP example: 5 jobs, 5 machines (all jobs follow M1->M2->M3->M4->M5)
J = np.array([
    [1, 2, 3, 4, 5],  # Job 0: M1 -> M2 -> M3 -> M4 -> M5
    [1, 2, 3, 4, 5],  # Job 1: M1 -> M2 -> M3 -> M4 -> M5
    [1, 2, 3, 4, 5],  # Job 2: same machine order
    [1, 2, 3, 4, 5],  # Job 3: same machine order
    [1, 2, 3, 4, 5]   # Job 4: same machine order
])
P = np.array([
    [5, 8, 3, 6, 4],  # Job 0 processing times
    [3, 7, 5, 9, 6],  # Job 1
    [6, 4, 8, 2, 7],  # Job 2
    [4, 9, 6, 5, 3],  # Job 3
    [7, 3, 4, 8, 5]   # Job 4
])

n, m = J.shape
print(f"Job数量: {n}")
print(f"工序数量: {m}")
print(f"机器数量: {J.max()}")
print(f"J (机器序列，所有Job相同):\n{J}")
print(f"P (加工时间):\n{P}")

# Run GA
best_chromosome, best_makespan, best_schedule = GA_FSP(
    J, P,
    popSize=100,
    generations=200,
    pc=0.8,
    pm=0.1
)